# Feature Engineering on the Titanic Dataset

This notebook demonstrates all major feature engineering techniques on the Titanic survival dataset using Python and scikit-learn.


## Checklist

1. Handle missing values
2. Handle categorical values
3. Scale the features
4. Remove outliers
5. Feature selection
6. PCA


## Step 1 — Load and Inspect the Dataset

We will use the Titanic dataset, which contains passenger information and whether they survived (`Survived`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml

# Load Titanic dataset directly from OpenML
titanic = fetch_openml('titanic', version=1, as_frame=True)
df = titanic.frame

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


## Step 2 — Handle Missing Values

We inspect missing counts per column and apply sensible fixes:

- **Age**: impute with median (robust to outliers)
- **Embarked**: impute with mode
- **Cabin**: too many missing values; we will drop it
- **Fare**: impute with median if any value is missing


In [ ]:
# Check missing value counts
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(missing)

# Missing percentage
print('\nMissing %:')
print((missing / len(df) * 100).round(2))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()


In [ ]:
# --- Handle missing values ---

# 1. Drop Cabin (too many missing values)
df = df.drop(columns=['cabin', 'boat', 'body', 'home.dest', 'name', 'ticket'])

# 2. Impute Age with median
age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)

# 3. Impute Fare with median
fare_median = df['fare'].median()
df['fare'] = df['fare'].fillna(fare_median)

# 4. Impute Embarked with mode
embarked_mode = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(embarked_mode)

print('After imputation:')
print(df.isnull().sum().sort_values(ascending=False))


## Step 3 — Handle Categorical Values

Machine learning models work with numbers, so we need to convert text categories into numeric forms:

- **One-Hot Encoding** for nominal variables like `Sex` and `Embarked`
- **Ordinal Encoding** if a natural order exists (not needed here)


In [ ]:
print('Unique values in Sex:', df['sex'].unique())
print('Unique values in Embarked:', df['embarked'].unique())


In [ ]:
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

print('DataFrame shape after encoding:', df.shape)
df.head()


## Step 4 — Remove Outliers

Outliers can distort model training. We detect them using the **IQR method** and remove extreme values from `Age` and `Fare`.


In [ ]:
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[column] >= lower) & (data[column] <= upper)]

print('Shape before outlier removal:', df.shape)
df = remove_outliers_iqr(df, 'age')
df = remove_outliers_iqr(df, 'fare')
print('Shape after outlier removal:', df.shape)


In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.boxplot(x=df['age'])
plt.title('Age Distribution After Outlier Removal')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['fare'])
plt.title('Fare Distribution After Outlier Removal')

plt.tight_layout()
plt.show()


## Step 5 — Scale the Features

Feature scaling ensures all numeric columns are on a comparable scale, which helps many algorithms perform better.

We use **StandardScaler**, which centers data around mean 0 with standard deviation 1.


In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['age', 'fare', 'pclass']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df[numerical_cols].describe().T


## Step 6 — Feature Selection

Not all features are equally useful. We use **SelectKBest** with chi-square to keep the top features most related to survival.

This reduces noise and improves model efficiency.


In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

# Separate features and target
target = df['survived']
features = df.drop(columns=['survived'])

# Select top 6 features
selector = SelectKBest(score_func=chi2, k=6)
X_selected = selector.fit_transform(features, target)

selected_features = features.columns[selector.get_support()]
print('Selected features:', list(selected_features))

# Show feature importance scores
scores = pd.Series(selector.scores_, index=features.columns)
print(scores.sort_values(ascending=False))


In [ ]:
# Visualization of feature scores
plt.figure(figsize=(8, 4))
scores.sort_values().plot(kind='barh')
plt.title('Feature Importance Scores (Chi-Square)')
plt.xlabel('Score')
plt.tight_layout()
plt.show()


## Step 7 — PCA (Principal Component Analysis)

PCA reduces high-dimensional data into fewer components while preserving as much variance as possible. This helps with visualization and removes multicollinearity.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
X_pca = pca.fit_transform(X_selected)

# Explained variance ratio
explained_var = pd.Series(
    pca.explained_variance_ratio_,
    index=[f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))]
)
print(explained_var)

print('\nCumulative variance explained:')
print(explained_var.cumsum().round(4))


In [ ]:
# Scree plot
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(explained_var)+1), explained_var, marker='o', linestyle='--')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot')
plt.grid(True)
plt.xticks(range(1, len(explained_var)+1))
plt.tight_layout()
plt.show()


In [ ]:
# Project data onto first 2 principal components for visualization
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_selected)

plt.figure(figsize=(8, 5))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=target, palette='coolwarm', alpha=0.7)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Titanic Data in 2D (PCA)')
plt.legend(title='Survived')
plt.grid(True)
plt.tight_layout()
plt.show()


## Summary of Feature Engineering Pipeline

| Step | Technique |
|------|-----------|
| 1 | **Missing Values**: Dropped `Cabin`, imputed `Age`/`Fare` with median, `Embarked` with mode |
| 2 | **Categorical Encoding**: One-Hot Encoding for `Sex` and `Embarked` |
| 3 | **Outlier Removal**: IQR-based filtering on `Age` and `Fare` |
| 4 | **Feature Scaling**: StandardScaler on numerical columns |
| 5 | **Feature Selection**: SelectKBest (chi-square) chose top 6 features |
| 6 | **PCA**: Reduced dimensions while preserving variance for visualization |

These steps produced a cleaner, more model-ready dataset suitable for classification tasks like predicting Titanic survival.
